# V2.1 — Guarded, Historically-Contacted Walk-Forward Readout

Route: `lst_models` V2.1 extension
Scope: `guarded_walkforward_readout` (holdout contacted under guarded designation; zero official-validation scoring events)
Designation: **guarded, historically-contacted walk-forward readout**

This notebook executes the pre-registered V2.1 walk-forward readout on the post-2017-01-25 closed segment. The holdout was contacted during V1 development; all results carry the guarded qualification per `docs/protocols/v2_1_guarded_walkforward_readout_protocol.md`.

FORBIDDEN wording in any output: clean test, clean holdout, untouched test, untouched holdout, final evidence, out-of-sample proof.

## Protocol Summary

- **Walk-forward design**: k=7 expanding-window periods, each ~12 months, starting at 2017-01-25. Period 7 truncated at data end (2024-04-19).
- **Model roster**: 5 models (Dummy, LightGBM, Std DLinear, TCN, MS-DLinear+TCN), all frozen params from Stage 02/03. Zero new HPO.
- **Refit recipe**: frozen D1/D2 mechanism — chronological-tail early stopping carved from refit rows only; scored period rows never participate.
- **Seeds**: `[101, 202]` (frozen V2 seed policy).
- **Stability criteria** (judged on `tcn_frozen_primary` only): `positive_period_count >= 2` AND `pooled_delta > 0`.
- **Total scoring events**: 7 periods x 4 model rows x 2 seeds = 56. Plus 1 coverage-probe metadata contact.
- **Period-boundary invalidation**: `invalid_cross_split` flags enforced at every period boundary; assertion logged in manifest.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import hashlib
import subprocess
import sys
import time

# ── Execution gates (all default OFF) ──────────────────────────────────
RUN_PROJECT_BOOTSTRAP = True
PROJECT_BOOTSTRAP_MODE = "github_commit"  # github_commit | manual_upload | already_present
RUN_UPSTREAM_DRIVE_SYNC = True
RUN_RAW_DATA_SYNC = True
RUN_V2_1 = False          # master gate for the walk-forward readout
RUN_DRIVE_BACKUP = True
RUN_ARTIFACT_DISPLAY = False

# ── Project coordinates ────────────────────────────────────────────────
PROJECT_REPO_URL = "https://github.com/zkc768/lstm-zhang.git"
PROJECT_REPO_COMMIT = "<FILL_WITH_FULL_BUNDLE_COMMIT>"
PROJECT_ROOT = Path("/content/lst_models")

# ── Upstream run IDs (frozen V2 chain) ─────────────────────────────────
STAGE_NAME = "v2_1_guarded_walkforward_readout"
ROUTE = "lst_models"
SCOPE = "guarded_walkforward_readout"
STAGE00_RUN_ID = "20260610_051705_347450"
STAGE01_RUN_ID = "20260610_075002"
STAGE02_RUN_ID = "20260610_082130_797479"
STAGE03_RUN_ID = "20260610_133305_716174"
STAGE04_RUN_ID = "20260610_232623_326133"
STAGE04_ORDERING = "stage04_first"
STAGE04_ORDERING_OVERRIDE_REASON = None
SUPERSEDED_STAGE02_RUN_IDS = ["20260609_100637_704705", "20260610_010019_507648"]
FROZEN_SEEDS = [101, 202]

# ── Paths ──────────────────────────────────────────────────────────────
STAGE00_OUTPUT_DIR = Path("/content/lst_models_results/00_data_split_label_freeze") / STAGE00_RUN_ID
STAGE01_OUTPUT_DIR = Path("/content/lst_models_results/01_feature_window_search") / STAGE01_RUN_ID
STAGE02_OUTPUT_DIR = Path("/content/lst_models_results/02_model_hpo_train_inner") / STAGE02_RUN_ID
STAGE03_OUTPUT_DIR = Path("/content/lst_models_results/03_frozen_validation_readout") / STAGE03_RUN_ID
OUTPUT_DIR = Path("/content/lst_models_results/v2_1_guarded_walkforward_readout")
RAW_DATA_DIR = Path("/content/lst_models_raw_stock_data")
CHECKPOINT_ROOT = Path("/content/lst_models_checkpoints/v2_1_guarded_walkforward_readout")
STAGE00_DRIVE_PATH_PARTS = ["lst_models", "results", "00_data_split_label_freeze", STAGE00_RUN_ID]
STAGE01_DRIVE_PATH_PARTS = ["lst_models", "results", "01_feature_window_search", STAGE01_RUN_ID]
STAGE02_DRIVE_PATH_PARTS = ["lst_models", "results", "02_model_hpo_train_inner", STAGE02_RUN_ID]
STAGE03_DRIVE_PATH_PARTS = ["lst_models", "results", "03_frozen_validation_readout", STAGE03_RUN_ID]
V2_1_DRIVE_RESULT_PATH_PARTS = ["lst_models", "results", "v2_1_guarded_walkforward_readout"]

# ── Walk-forward period table ──────────────────────────────────────────
PERIODS = [
    {"period_id": "wf_p1", "start": "2017-01-25", "end_exclusive": "2018-01-25"},
    {"period_id": "wf_p2", "start": "2018-01-25", "end_exclusive": "2019-01-25"},
    {"period_id": "wf_p3", "start": "2019-01-25", "end_exclusive": "2020-01-25"},
    {"period_id": "wf_p4", "start": "2020-01-25", "end_exclusive": "2021-01-25"},
    {"period_id": "wf_p5", "start": "2021-01-25", "end_exclusive": "2022-01-25"},
    {"period_id": "wf_p6", "start": "2022-01-25", "end_exclusive": "2023-01-25"},
    {"period_id": "wf_p7", "start": "2023-01-25", "end_exclusive": "2024-04-19"},
]

# ── Model roster (frozen params from Stage 02/03) ─────────────────────
MODEL_ROSTER = [
    {
        "table_row_id": "tcn_frozen_primary",
        "model_family": "tcn",
        "probe_id": "tcn_tiny",
        "hpo_profile_id": "tcn_p01",
        "params": {"channels": [16, 16], "kernel_size": 2, "dropout": 0.0,
                   "learning_rate": 0.001, "weight_decay": 0.0001},
    },
    {
        "table_row_id": "lightgbm_family_best",
        "model_family": "lightgbm",
        "probe_id": "lightgbm_small",
        "hpo_profile_id": "lgbm_p03",
        "params": {"n_estimators": 300, "learning_rate": 0.02, "max_depth": 8,
                   "num_leaves": 63, "min_child_samples": 100, "subsample": 0.9,
                   "colsample_bytree": 0.9, "reg_lambda": 5.0, "class_weight": "balanced"},
    },
    {
        "table_row_id": "standard_dlinear_family_best",
        "model_family": "standard_dlinear",
        "probe_id": "standard_dlinear_tiny",
        "hpo_profile_id": "dlinear_p03",
        "params": {"moving_avg_kernel": 7, "dropout": 0.10,
                   "learning_rate": 0.0003, "weight_decay": 0.0001},
    },
    {
        "table_row_id": "ms_dlinear_tcn_family_best",
        "model_family": "ms_dlinear_tcn",
        "probe_id": "ms_dlinear_tcn_tiny",
        "hpo_profile_id": "msdt_p01",
        "params": {"moving_avg_kernels": [3, 5, 9], "tcn_channels": [16, 16],
                   "tcn_kernel_size": 3, "dropout": 0.10,
                   "learning_rate": 0.001, "weight_decay": 0.0001},
    },
]

CANDIDATE_ID = "price_volume_time_w20"
FEATURE_SET = "price_volume_time"
WINDOW_SIZE = 20
LABEL_HORIZON_K = 9
LABEL_BAND_BPS = 3.0

V2_1_SIGN_OFF = {"status": "pending", "user_sign_off_date": None, "advisor_confirmation_reference": None, "resolved_open_decisions": {"OD-A_period_count_k": len(PERIODS), "OD-B_period_length_months": 12, "OD-C_candidate_input_policy": "A_family_best_verbatim", "OD-D_ablation_rows_included": False, "OD-E_stage04_ordering": STAGE04_ORDERING, "OD-F_criteria_accepted": "yes"}}
V2_1_COVERAGE_PROBE = {"authorization": "pending", "artifact": "v2_1_coverage_probe.json", "artifact_sha256": None, "probe_timestamp_utc": None}

# ── Bootstrap ──────────────────────────────────────────────────────────
def run_cmd(args, cwd=None):
    print("+", " ".join(str(a) for a in args))
    subprocess.run(args, cwd=cwd, check=True)

if RUN_PROJECT_BOOTSTRAP:
    if PROJECT_BOOTSTRAP_MODE == "github_commit":
        if "<" in PROJECT_REPO_COMMIT:
            raise ValueError("fill PROJECT_REPO_COMMIT before bootstrapping")
        if (PROJECT_ROOT / ".git").exists():
            run_cmd(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        else:
            run_cmd(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)])
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        actual = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
        ).strip()
        assert actual == PROJECT_REPO_COMMIT, (actual, PROJECT_REPO_COMMIT)
    elif PROJECT_BOOTSTRAP_MODE == "already_present":
        pass
    else:
        raise ValueError("unsupported PROJECT_BOOTSTRAP_MODE")

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

cached = [n for n in sys.modules if n == "lst_models" or n.startswith("lst_models.")]
for n in cached:
    del sys.modules[n]
importlib.invalidate_caches()

STAGE_CONFIG_PATH = PROJECT_ROOT / "configs" / "stages" / "v2_1_guarded_walkforward_readout.yaml"
PROTOCOL_PATH = PROJECT_ROOT / "docs" / "protocols" / "v2_1_guarded_walkforward_readout_protocol.md"
RAW_DATA_MANIFEST_PATH = PROJECT_ROOT / "configs" / "lst_models_data.yaml"

for p in [STAGE_CONFIG_PATH, PROTOCOL_PATH, RAW_DATA_MANIFEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"missing: {p}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("STAGE_NAME:", STAGE_NAME)
print("PERIODS:", len(PERIODS))
print("MODEL_ROSTER:", [m["table_row_id"] for m in MODEL_ROSTER])
print("RUN_V2_1:", RUN_V2_1)

## Config Load And Contract Check

In [ ]:
import yaml

with STAGE_CONFIG_PATH.open("r", encoding="utf-8") as fh:
    v2_1_config = yaml.safe_load(fh)

v2_1_config["inputs"]["stage04_run_id"] = STAGE04_RUN_ID
v2_1_config["inputs"]["stage04_ordering"] = STAGE04_ORDERING
v2_1_config["inputs"]["stage04_ordering_override_reason"] = STAGE04_ORDERING_OVERRIDE_REASON
v2_1_config["sign_off"] = V2_1_SIGN_OFF
v2_1_config["coverage_probe"] = V2_1_COVERAGE_PROBE

assert v2_1_config["stage_name"] == STAGE_NAME
assert v2_1_config["route"] == ROUTE
assert v2_1_config["scope"] == SCOPE
assert v2_1_config["holdout_test_contact"] is True
assert v2_1_config["holdout_contact_tier"] == "guarded_historically_contacted"
assert v2_1_config["clean_test_claim"] is False
assert v2_1_config["official_validation_scoring_events"] == 0
assert v2_1_config["no_final_model_selected"] is True
assert v2_1_config["v2_frozen_selection_unchanged"] is True
assert v2_1_config["inputs"]["stage04_ordering"] == "stage04_first"
assert v2_1_config["sign_off"]["status"] in {"pending", "complete"}
assert v2_1_config["coverage_probe"]["artifact"] == "v2_1_coverage_probe.json"

wf = v2_1_config["walkforward"]
assert wf["seeds"] == FROZEN_SEEDS
assert wf["period_count"] == len(PERIODS)
for i, p in enumerate(wf["periods"]):
    assert p["period_id"] == PERIODS[i]["period_id"]
    assert p["start"] == PERIODS[i]["start"]
    assert p["end_exclusive"] == PERIODS[i]["end_exclusive"]

roster = v2_1_config["model_roster"]
config_rows = {r["table_row_id"]: r for r in roster["model_rows"]}
for model in MODEL_ROSTER:
    row = config_rows[model["table_row_id"]]
    assert row["model_family"] == model["model_family"]
    assert row["probe_id"] == model["probe_id"]
    assert row["hpo_profile_id"] == model["hpo_profile_id"]

lgbm_defaults = v2_1_config["lightgbm_training_defaults"]
assert lgbm_defaults["early_stopping_validation_source"] == "inner_train_chronological_tail"
assert int(lgbm_defaults["early_stopping_rounds"]) == 25

torch_defaults = v2_1_config["probe_training_defaults"]["torch"]
assert torch_defaults["early_stopping"] == "inner_train_chronological_tail"
assert int(torch_defaults["early_stopping_patience"]) == 8
assert int(torch_defaults["epochs"]) == 64

assert v2_1_config["forbidden"]["wording"], "forbidden wording list must be non-empty"
assert "clean test" in v2_1_config["forbidden"]["wording"]

print("Config contract check passed.")
print(json.dumps({
    "stage_name": v2_1_config["stage_name"],
    "scope": v2_1_config["scope"],
    "holdout_contact_tier": v2_1_config["holdout_contact_tier"],
    "period_count": wf["period_count"],
    "model_rows": [r["table_row_id"] for r in roster["model_rows"]],
}, indent=2))

## Upstream Inputs: Drive Sync And Artifact Verification

In [ ]:
def quote_drive_query_value(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")

def get_drive_service():
    from google.colab import auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    return build("drive", "v3")

def find_unique_drive_child(service, parent_id, name, mime_type=None):
    escaped = quote_drive_query_value(name)
    q = [f"name = '{escaped}'", f"'{parent_id}' in parents", "trashed = false"]
    if mime_type:
        q.append(f"mimeType = '{mime_type}'")
    resp = service.files().list(q=" and ".join(q), spaces="drive",
                                fields="files(id, name, mimeType, size)", pageSize=10).execute()
    matches = resp.get("files", [])
    if len(matches) != 1:
        raise FileNotFoundError(f"expected 1 Drive item {name!r}; found {len(matches)}")
    return matches[0]

def resolve_drive_folder(service, path_parts):
    fid = "root"
    for part in path_parts:
        fid = find_unique_drive_child(service, fid, part, "application/vnd.google-apps.folder")["id"]
    return fid

def download_drive_file(service, file_id, output_path):
    from googleapiclient.http import MediaIoBaseDownload
    output_path.parent.mkdir(parents=True, exist_ok=True)
    req = service.files().get_media(fileId=file_id)
    with output_path.open("wb") as fh:
        dl = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            _, done = dl.next_chunk()

def ensure_drive_folder(service, parent_id, name):
    mime = "application/vnd.google-apps.folder"
    escaped = quote_drive_query_value(name)
    q = f"name = '{escaped}' and '{parent_id}' in parents and trashed = false and mimeType = '{mime}'"
    matches = service.files().list(q=q, fields="files(id)", pageSize=10).execute().get("files", [])
    if len(matches) == 1:
        return matches[0]["id"]
    if len(matches) > 1:
        raise RuntimeError(f"duplicate Drive folders: {name!r}")
    return service.files().create(body={"name": name, "mimeType": mime, "parents": [parent_id]},
                                  fields="id").execute()["id"]

def ensure_drive_path(service, path_parts):
    fid = "root"
    for part in path_parts:
        fid = ensure_drive_folder(service, fid, part)
    return fid

def upload_or_update_drive_file(service, parent_id, local_path):
    from googleapiclient.http import MediaFileUpload
    name = local_path.name
    escaped = quote_drive_query_value(name)
    q = f"name = '{escaped}' and '{parent_id}' in parents and trashed = false"
    matches = service.files().list(q=q, fields="files(id, name)", pageSize=10).execute().get("files", [])
    media = MediaFileUpload(str(local_path), resumable=True)
    if not matches:
        return service.files().create(body={"name": name, "parents": [parent_id]},
                                      media_body=media, fields="id, name, size").execute()
    if len(matches) == 1:
        return service.files().update(fileId=matches[0]["id"], media_body=media,
                                      fields="id, name, size").execute()
    raise RuntimeError(f"duplicate Drive files: {name!r}")

def sync_stage_artifacts(service, output_dir, path_parts, required, label):
    fid = resolve_drive_folder(service, path_parts)
    for name in required:
        out = Path(output_dir) / name
        if out.exists():
            continue
        meta = find_unique_drive_child(service, fid, name)
        download_drive_file(service, meta["id"], out)
        print(f"downloaded {label}: {name}")

def sync_raw_data(service):
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as fh:
        manifest = yaml.safe_load(fh)
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for ticker in manifest["tickers"]:
        spec = manifest["raw_source"]["files"][ticker]
        out = RAW_DATA_DIR / spec["name"]
        if out.exists():
            continue
        download_drive_file(service, spec["file_id"], out)
        print(f"downloaded raw: {out.name}")

required_s00 = v2_1_config["inputs"]["required_stage00_artifacts"]
required_s01 = v2_1_config["inputs"]["required_stage01_artifacts"]
required_s02 = v2_1_config["inputs"]["required_stage02_artifacts"]
required_s03 = v2_1_config["inputs"]["required_stage03_artifacts"]

if RUN_V2_1:
    from lst_models.stages.guarded_walkforward_readout import _validate_config
    _validate_config(v2_1_config)
    service = get_drive_service()
    if RUN_RAW_DATA_SYNC:
        sync_raw_data(service)
    if RUN_UPSTREAM_DRIVE_SYNC:
        sync_stage_artifacts(service, STAGE00_OUTPUT_DIR, STAGE00_DRIVE_PATH_PARTS, required_s00, "s00")
        sync_stage_artifacts(service, STAGE01_OUTPUT_DIR, STAGE01_DRIVE_PATH_PARTS, required_s01, "s01")
        sync_stage_artifacts(service, STAGE02_OUTPUT_DIR, STAGE02_DRIVE_PATH_PARTS, required_s02, "s02")
        sync_stage_artifacts(service, STAGE03_OUTPUT_DIR, STAGE03_DRIVE_PATH_PARTS, required_s03, "s03")
    from lst_models.artifacts import require_artifacts
    require_artifacts(STAGE00_OUTPUT_DIR, required_s00)
    require_artifacts(STAGE01_OUTPUT_DIR, required_s01)
    require_artifacts(STAGE02_OUTPUT_DIR, required_s02)
    require_artifacts(STAGE03_OUTPUT_DIR, required_s03)
    print("All upstream artifacts verified.")
else:
    print("RUN_V2_1=False; upstream sync skipped.")

## Execute Walk-Forward Readout (via `run_stage`)

The runner module `src/lst_models/stages/guarded_walkforward_readout.py` handles data loading, walk-forward loop, refit, scoring, comparison table, stability judgement, and all artifact writes. The notebook calls `run_stage(config)` and receives a `V21Result` dataclass.

In [ ]:
if RUN_V2_1:
    from lst_models.stages.guarded_walkforward_readout import run_stage

    result = run_stage(v2_1_config)

    run_id = result.output_dir.name
    run_dir = result.output_dir

    print(f"RUN_ID: {run_id}")
    print(f"RUN_DIR: {run_dir}")
    print(f"Manifest: {result.run_manifest}")
    print(f"Decision: {result.decision_record}")
    print()
    for f in sorted(run_dir.iterdir()):
        print(f"  {f.name} ({f.stat().st_size:,} bytes)")
else:
    print("RUN_V2_1=False; walk-forward readout skipped.")

## Walk-Forward Results: Comparison Table And Period Summary

In [ ]:
if RUN_V2_1:
    import pandas as pd

    print("=== Ian's Comparison Table (Guarded Walk-Forward Readout) ===")
    comparison_df = pd.read_csv(result.comparison_table)
    display(comparison_df)

    print("\n=== Period Summary (mean over seeds, per period x model) ===")
    period_df = pd.read_csv(result.period_summary)
    display(period_df)
else:
    print("RUN_V2_1=False; results display skipped.")

## Stability Criteria And Decision Record

In [ ]:
if RUN_V2_1:
    decision_data = json.loads(result.decision_record.read_text(encoding="utf-8"))

    print("=== Stability Criteria (tcn_frozen_primary) ===")
    print(f"  positive_period_count = {decision_data['positive_period_count']}"
          f" (>= 2: {decision_data['criteria_met']['positive_period_count']})")
    print(f"  pooled_delta = {decision_data['pooled_delta']:.6f}"
          f" (> 0: {decision_data['criteria_met']['pooled_delta']})")
    print(f"  DECISION: {decision_data['decision']}")
    print(f"  readout_complete: {decision_data['readout_complete']}")
    print(f"  guarded_scoring_events: {decision_data['guarded_scoring_events']}")
else:
    print("RUN_V2_1=False; decision display skipped.")

## Durable Drive Result Save

In [ ]:
if RUN_DRIVE_BACKUP and RUN_V2_1:
    decision_data = json.loads(result.decision_record.read_text(encoding="utf-8"))
    manifest_data = json.loads(result.run_manifest.read_text(encoding="utf-8"))
    if manifest_data.get("holdout_contact_tier") != "guarded_historically_contacted":
        raise ValueError("backup refused: missing guarded tier")
    if manifest_data.get("clean_test_claim") is not False:
        raise ValueError("backup refused: clean_test_claim must be false")

    drive_path_parts = V2_1_DRIVE_RESULT_PATH_PARTS + [run_id]
    drive_folder_id = ensure_drive_path(service, drive_path_parts)

    local_files = sorted(p for p in run_dir.rglob("*") if p.is_file())
    uploads = []
    for path in local_files:
        uploaded = upload_or_update_drive_file(service, drive_folder_id, path)
        print(f"  uploaded: {path.name}")
        uploads.append({"name": path.name, "bytes": path.stat().st_size})

    backup_manifest = {
        "stage_name": STAGE_NAME, "run_id": run_id,
        "decision": decision_data["decision"],
        "guarded_scoring_events": decision_data["guarded_scoring_events"],
        "drive_path": "My Drive/" + "/".join(drive_path_parts),
        "uploaded_files": uploads,
        "sync_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "holdout_contact_tier": "guarded_historically_contacted",
        "clean_test_claim": False,
    }
    backup_path = run_dir / "drive_backup_manifest.json"
    backup_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    upload_or_update_drive_file(service, drive_folder_id, backup_path)

    print(f"\nDrive backup complete: My Drive/{'/'.join(drive_path_parts)}")
else:
    print("Drive backup skipped.")

## Readout Display

Allowed wording: `guarded walk-forward evidence`, `guarded period-stability readout`, `primary candidate met/did not meet predeclared guarded stability criteria`. Do not claim a final model, clean test, or out-of-sample proof from this notebook.

In [ ]:
if RUN_ARTIFACT_DISPLAY and RUN_V2_1:
    import pandas as pd

    decision_data = json.loads(result.decision_record.read_text(encoding="utf-8"))
    print("=== Decision Record Summary ===")
    print(json.dumps({
        "decision": decision_data["decision"],
        "positive_period_count": decision_data["positive_period_count"],
        "pooled_delta": decision_data["pooled_delta"],
        "criteria_met": decision_data["criteria_met"],
        "guarded_scoring_events": decision_data["guarded_scoring_events"],
        "readout_complete": decision_data["readout_complete"],
    }, indent=2))

    print("\n=== Comparison Table ===")
    display(pd.read_csv(result.comparison_table))

    print("\n=== Period Summary ===")
    display(pd.read_csv(result.period_summary))

    print("\n=== Full Readout (first 20 rows) ===")
    display(pd.read_csv(result.walkforward_readout).head(20))

    print("\n=== Baselines ===")
    display(pd.read_csv(result.same_row_baselines))
else:
    print("RUN_ARTIFACT_DISPLAY=False or RUN_V2_1=False; display skipped.")